In [14]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import EuroSAT
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms

In [15]:
class ResNet50_M3(nn.Module):
    def __init__(self):
        super().__init__()

        # Load the heavier ResNet-50 model
        self.model = resnet50(weights=ResNet50_Weights.DEFAULT)

        # Fine-tune everything initially
        for param in self.model.parameters():
            param.requires_grad = True

        # Freeze BatchNorm2d parameters to keep pre-trained image statistics stable
        for module in self.model.modules():
            if isinstance(module, nn.BatchNorm2d):
                for param in module.parameters():
                    param.requires_grad = False

        # Replace the final classification head for our 10 EuroSAT classes
        self.model.fc = nn.Sequential(
            nn.Linear(self.model.fc.in_features, 200),
            nn.ReLU(),
            nn.Dropout(p=0.3), # Increased dropout slightly to prevent overfitting

            nn.Linear(200, 100),
            nn.ReLU(),
            nn.Dropout(p=0.3),

            nn.Linear(100, 10)
        )

    def train(self, mode=True):
        super().train(mode)
        # Keep BatchNorm layers in evaluation mode during training
        for module in self.model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()
        return self

    def forward(self, x):
        return self.model(x)

In [16]:
BATCH_SIZE = 64
EPOCHS = 25 # Slightly higher max epochs because the scheduler will help us train longer safely
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs("/content/reports", exist_ok=True)
os.makedirs("/content/checkpoints", exist_ok=True)
os.makedirs("/content/data/processed", exist_ok=True)

In [17]:
weights = ResNet50_Weights.DEFAULT

# Advanced Augmentation Pipeline
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=45), # New: Rotate images up to 45 degrees
    weights.transforms()
])

dataset = EuroSAT(
    root="/content/data/raw",
    download=True,
    transform=transform
)

In [18]:
with open("/content/data/processed/splits.json") as f:
    splits = json.load(f)

train_dataset = Subset(dataset, splits["train"])
val_dataset   = Subset(dataset, splits["val"])
test_dataset  = Subset(dataset, splits["test"])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
model = ResNet50_M3().to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

# NEW: The Learning Rate Scheduler
# If validation loss doesn't improve for 2 epochs, cut the learning rate in half
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 211MB/s]


In [21]:
PATIENCE = 6 # Increased patience since the scheduler needs time to work
best_val_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_accuracy = correct / total

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            predictions = outputs.argmax(dim=1)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_accuracy = val_correct / val_total

    # Step the scheduler based on validation loss
    scheduler.step(val_loss)

    # Save logic
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
    }

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(checkpoint, "/content/checkpoints/resnet50_m3_best.pth")
        print(f"Epoch [{epoch + 1}/{EPOCHS}] Train Acc: {train_accuracy * 100:.2f}% | Val Acc: {val_accuracy * 100:.2f}% ← best")
    else:
        epochs_without_improvement += 1
        print(f"Epoch [{epoch + 1}/{EPOCHS}] Train Acc: {train_accuracy * 100:.2f}% | Val Acc: {val_accuracy * 100:.2f}% ({epochs_without_improvement}/{PATIENCE})")
        if epochs_without_improvement >= PATIENCE:
            print(f"\nEarly stopping triggered after {epoch + 1} epochs.")
            break

# Restore best model
best_checkpoint = torch.load("/content/checkpoints/resnet50_m3_best.pth", map_location=DEVICE)
model.load_state_dict(best_checkpoint["model_state_dict"])
print(f"\nBest ResNet-50 restored from epoch {best_checkpoint['epoch']} with validation accuracy {best_checkpoint['val_accuracy'] * 100:.2f}%")

Epoch [1/25] Train Acc: 79.33% | Val Acc: 92.84% ← best
Epoch [2/25] Train Acc: 93.72% | Val Acc: 94.44% ← best
Epoch [3/25] Train Acc: 95.15% | Val Acc: 94.35% (1/6)
Epoch [4/25] Train Acc: 96.09% | Val Acc: 94.86% ← best
Epoch [5/25] Train Acc: 96.24% | Val Acc: 96.49% ← best
Epoch [6/25] Train Acc: 96.70% | Val Acc: 95.90% (1/6)
Epoch [7/25] Train Acc: 96.71% | Val Acc: 97.23% ← best
Epoch [8/25] Train Acc: 97.42% | Val Acc: 96.27% (1/6)
Epoch [9/25] Train Acc: 97.25% | Val Acc: 96.77% (2/6)
Epoch [10/25] Train Acc: 97.48% | Val Acc: 94.15% (3/6)
Epoch [11/25] Train Acc: 98.56% | Val Acc: 97.70% ← best
Epoch [12/25] Train Acc: 98.72% | Val Acc: 97.60% (1/6)
Epoch [13/25] Train Acc: 98.67% | Val Acc: 97.73% (2/6)
Epoch [14/25] Train Acc: 98.62% | Val Acc: 97.95% ← best
Epoch [15/25] Train Acc: 98.63% | Val Acc: 98.05% ← best
Epoch [16/25] Train Acc: 98.51% | Val Acc: 97.78% (1/6)
Epoch [17/25] Train Acc: 98.77% | Val Acc: 96.00% (2/6)
Epoch [18/25] Train Acc: 98.79% | Val Acc: 97.09%

In [22]:
# TEST CELL
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        predictions = outputs.argmax(dim=1)
        y_true.extend(labels.numpy())
        y_pred.extend(predictions.cpu().numpy())

# REPORT CELL
report = classification_report(y_true, y_pred, target_names=dataset.classes)
cm = confusion_matrix(y_true, y_pred)

print("\nClassification Report (ResNet-50 Advanced):")
print(report)
print("\nConfusion Matrix:")
print(cm)

with open("/content/reports/resnet50_advanced_report.txt", "w") as f:
    f.write(report)
    f.write("\n\nConfusion Matrix:\n")
    f.write(str(cm))


Classification Report (ResNet-50 Advanced):
                      precision    recall  f1-score   support

          AnnualCrop       0.97      0.97      0.97       450
              Forest       0.99      1.00      0.99       450
HerbaceousVegetation       0.99      0.96      0.97       450
             Highway       0.99      0.98      0.98       375
          Industrial       0.99      0.99      0.99       375
             Pasture       0.97      0.99      0.98       300
       PermanentCrop       0.97      0.96      0.97       375
         Residential       1.00      0.99      0.99       450
               River       0.98      0.99      0.99       375
             SeaLake       0.99      1.00      0.99       450

            accuracy                           0.98      4050
           macro avg       0.98      0.98      0.98      4050
        weighted avg       0.98      0.98      0.98      4050


Confusion Matrix:
[[438   0   0   1   0   1   7   0   1   2]
 [  0 450   0   0   0 